<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_btm_karpathy_sts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT BTM — Branch-Train-Merge slider demo (Shakespeare + TinyStories char)

The cleaner alternative to the dual-slot scheme. Each slot trains as a standard nanoGPT char-shake model, on its own corpus, from a byte-identical shared init. No alpha-mix at training time. No mask_grads. At inference we merge two ways and expose each as a slider:

1. **Naive linear interp**: `W_mix = alpha*W_a + (1-alpha)*W_b`. Model-soup / task-arithmetic style. Bets on linear-mode connectivity from the shared init.
2. **Git Re-Basin interp**: first permute W_b's MLP inner-dim units per layer to align with W_a (Hungarian algorithm on the inner-product cost), THEN linearly interpolate. Robust to permutation-related basin drift, at the cost of some computation per merge.

Per-slot architecture matches Karpathy's `train_shakespeare_char.py`: `n_layer=6, n_head=6, n_embd=384, dropout=0.2, lr=1e-3, batch=64, max_iters=5000`. Each slot trains at Karpathy's full per-iter rate. Total compute on T4: ~90 min for both slots + ~5 min for the merge demo + plot.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet scipy

In [ ]:
import os, sys
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT
if '/content/abcGPT' not in sys.path:
    sys.path.insert(0, '/content/abcGPT')

In [ ]:
!python data/shakespeare_tinystories_char/prepare.py

## Data: load train.bin, find separator, build per-corpus batch loaders

In [ ]:
import os, pickle
import numpy as np
import torch

data_dir = '/content/abcGPT/data/shakespeare_tinystories_char'
train_arr = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
val_arr   = np.memmap(os.path.join(data_dir, 'val.bin'),   dtype=np.uint16, mode='r')
with open(os.path.join(data_dir, 'meta.pkl'), 'rb') as f:
    meta = pickle.load(f)
vocab_size = meta['vocab_size']
stoi, itos = meta['stoi'], meta['itos']
print(f'vocab_size={vocab_size}  train chars={len(train_arr):,}  val chars={len(val_arr):,}')

# Locate the separator '\n\n===\n\n' to split train.bin into shake / ts ranges
sep_str = '\n\n===\n\n'
sep_ids = np.array([stoi[c] for c in sep_str], dtype=np.uint16)
# np.where scan for the first byte, then full-match verify
candidates = np.where(train_arr[:len(train_arr) - len(sep_ids) + 1] == sep_ids[0])[0]
sep_idx = None
for c in candidates:
    if np.array_equal(train_arr[c:c+len(sep_ids)], sep_ids):
        sep_idx = int(c); break
assert sep_idx is not None, 'separator not found'
shake_end = sep_idx
ts_start  = sep_idx + len(sep_ids)
ts_end    = len(train_arr)
print(f'shake range: [0:{shake_end}] ({shake_end:,} chars)')
print(f'ts    range: [{ts_start}:{ts_end}] ({ts_end - ts_start:,} chars)')

In [ ]:
block_size = 256
batch_size = 64
device = 'cuda'

def make_get_batch(arr, start, end):
    span = end - start
    def get_batch():
        ix = torch.randint(span - block_size, (batch_size,)) + start
        x = torch.stack([torch.from_numpy(arr[i:i+block_size].astype(np.int64)) for i in ix.tolist()])
        y = torch.stack([torch.from_numpy(arr[i+1:i+1+block_size].astype(np.int64)) for i in ix.tolist()])
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)
    return get_batch

# Per-corpus train loaders
shake_get_batch = make_get_batch(train_arr, 0, shake_end)
ts_get_batch    = make_get_batch(train_arr, ts_start, ts_end)

# Per-corpus val loaders (last 10% of each train range — small leakage but consistent across runs)
def val_range(start, end, frac=0.1):
    span = end - start
    return end - int(span * frac), end
shake_vs, shake_ve = val_range(0, shake_end)
ts_vs,    ts_ve    = val_range(ts_start, ts_end)
shake_val_batch = make_get_batch(train_arr, shake_vs, shake_ve)
ts_val_batch    = make_get_batch(train_arr, ts_vs,    ts_ve)
print(f'shake val: [{shake_vs}:{shake_ve}]  ts val: [{ts_vs}:{ts_ve}]')

## Build two models from byte-identical init

The shared starting point is the prerequisite for Linear Mode Connectivity. We init `model_a`, snapshot its state_dict, then load that into `model_b` to guarantee byte-equality (more robust than re-seeding, since any intervening RNG draw would diverge the inits).

In [ ]:
from btm_lib import GPT, GPTConfig, train_one_corpus, merge_naive, merge_rebasin, align_rebasin_mlp

config = GPTConfig(vocab_size=vocab_size, n_layer=6, n_head=6, n_embd=384,
                   block_size=block_size, dropout=0.2, bias=False)

torch.manual_seed(1337)
model_a = GPT(config).to(device)
init_sd = {k: v.detach().clone() for k, v in model_a.state_dict().items()}
model_b = GPT(config).to(device)
model_b.load_state_dict(init_sd)

# Verify byte-identical init
for (na, pa), (nb, pb) in zip(model_a.named_parameters(), model_b.named_parameters()):
    assert torch.equal(pa, pb), f'init mismatch at {na}'
n_params = sum(p.numel() for p in model_a.parameters())
print(f'two models built, byte-identical init  |  {n_params/1e6:.2f}M params each  |  {n_params*2/1e6:.2f}M total stored')

## Train slot A on tinyshakespeare

Standard Karpathy nanoGPT char-shake recipe, 5000 iters, ~46 min on T4.

In [ ]:
train_one_corpus(
    model_a, shake_get_batch,
    n_iters=5000, lr=1e-3, warmup=100, lr_decay_iters=5000, min_lr=1e-4,
    beta2=0.99, weight_decay=0.1, grad_clip=1.0,
    log_interval=250, eval_interval=500, eval_iters=200,
    get_val_batch_fn=shake_val_batch,
    device=device, amp_dtype=torch.bfloat16,
)

## Train slot B on TinyStories (same iter budget, byte-identical init)

In [ ]:
train_one_corpus(
    model_b, ts_get_batch,
    n_iters=5000, lr=1e-3, warmup=100, lr_decay_iters=5000, min_lr=1e-4,
    beta2=0.99, weight_decay=0.1, grad_clip=1.0,
    log_interval=250, eval_interval=500, eval_iters=200,
    get_val_batch_fn=ts_val_batch,
    device=device, amp_dtype=torch.bfloat16,
)

In [ ]:
# Save both slot checkpoints to Drive so the merge / sampling cells can be re-run without retraining
ckpt_dir = '/content/drive/MyDrive/abcGPT/btm_karpathy_sts'
os.makedirs(ckpt_dir, exist_ok=True)
torch.save({'model_state': model_a.state_dict(), 'config': config.__dict__}, f'{ckpt_dir}/slot_shake.pt')
torch.save({'model_state': model_b.state_dict(), 'config': config.__dict__}, f'{ckpt_dir}/slot_ts.pt')
print(f'saved to {ckpt_dir}')

## Merge sweep: val loss across alpha for both methods

At each alpha we build a fresh model from the merged state_dict and evaluate val loss on both corpora. **Naive** is the straight linear interp. **Re-Basin** first permutes B's MLP inner-dim units per layer to maximize inner-product alignment with A (via the Hungarian algorithm on `-W_fc_a @ W_fc_b.T`), then linearly interpolates the aligned versions. If LMC holds between A and B, naive should already be smooth and Re-Basin should be ~equivalent. If the slots have drifted into permutation-related basins, naive will show a high-loss barrier at mid-alpha that Re-Basin smooths out.

In [ ]:
sd_a = model_a.state_dict()
sd_b = model_b.state_dict()
n_layer = config.n_layer

# Sanity: endpoints recover original slots
_s = merge_naive(sd_a, sd_b, alpha=1.0)
assert all(torch.allclose(_s[k], sd_a[k]) for k in sd_a), 'naive alpha=1 should give sd_a'
_s = merge_naive(sd_a, sd_b, alpha=0.0)
assert all(torch.allclose(_s[k], sd_b[k]) for k in sd_b), 'naive alpha=0 should give sd_b'
print('endpoint sanity passes')

def make_merged_model(sd):
    m = GPT(config).to(device)
    m.load_state_dict(sd)
    return m

@torch.no_grad()
def eval_loss(model, get_batch_fn, n_iters=50):
    model.eval()
    losses = []
    for _ in range(n_iters):
        X, Y = get_batch_fn()
        with torch.amp.autocast(device_type=device, dtype=torch.bfloat16):
            _, loss = model(X, Y)
        losses.append(loss.item())
    return float(np.mean(losses))

alphas = np.linspace(0.0, 1.0, 11)
rows = []
for a in alphas:
    sd_n = merge_naive(sd_a, sd_b, alpha=float(a))
    m = make_merged_model(sd_n); n_sh = eval_loss(m, shake_val_batch); n_ts = eval_loss(m, ts_val_batch); del m
    sd_r = merge_rebasin(sd_a, sd_b, alpha=float(a), n_layer=n_layer)
    m = make_merged_model(sd_r); r_sh = eval_loss(m, shake_val_batch); r_ts = eval_loss(m, ts_val_batch); del m
    rows.append((a, n_sh, n_ts, r_sh, r_ts))
    print(f'alpha={a:.2f}  naive: shake={n_sh:.3f} ts={n_ts:.3f}   rebasin: shake={r_sh:.3f} ts={r_ts:.3f}')

rows = np.array(rows)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rows[:,0], rows[:,1], 'o-', label='naive')
axes[0].plot(rows[:,0], rows[:,3], 's-', label='re-basin')
axes[0].set_xlabel('alpha (1.0 = shake slot)')
axes[0].set_ylabel('val loss on tinyshakespeare')
axes[0].set_title('shake val')
axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].plot(rows[:,0], rows[:,2], 'o-', label='naive')
axes[1].plot(rows[:,0], rows[:,4], 's-', label='re-basin')
axes[1].set_xlabel('alpha (1.0 = shake slot)')
axes[1].set_ylabel('val loss on TinyStories')
axes[1].set_title('tinystories val')
axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

## Generate text at alpha in {0, 0.25, 0.5, 0.75, 1.0}, both merge methods

Side-by-side qualitative check. If naive degrades at mid-alpha and re-basin holds, that's the LMC-failure signature.

In [ ]:
def encode(s):
    return [stoi[c] for c in s if c in stoi]
def decode(ids):
    return ''.join(itos[i] for i in ids)

prompt = 'the king of '
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print('=' * 70)
    print(f'alpha = {a}')
    for label, sd_m in [
        ('naive  ', merge_naive(sd_a, sd_b, alpha=a)),
        ('rebasin', merge_rebasin(sd_a, sd_b, alpha=a, n_layer=n_layer)),
    ]:
        m = make_merged_model(sd_m); m.eval()
        torch.manual_seed(0)
        out = m.generate(prompt_ids, max_new_tokens=200, temperature=0.8, top_k=40)
        print(f'--- {label} ---')
        print(decode(out[0].tolist()))
        del m